# How reward terms shape a quadruped gait (Unitree Go1, MuJoCo Playground)

Kaggle version of the study. Settings: **Accelerator = GPU T4 x2**, **Internet = on**.
Best way to run: **Save Version → Save & Run All (Commit)**, so it keeps running after you close the tab. Results are written to `/kaggle/working` (Output tab).

In [ ]:
# Install (versions that work together with JAX 0.11)
!pip install -q mujoco mujoco_mjx brax "playground==0.2.0" "flax>=0.12.12" mediapy
!command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -qq ffmpeg)

In [ ]:
# GPU + rendering setup (from the official notebook)

import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.'
  )

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco

  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".'
  )

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

In [ ]:
# @title Import packages for plotting and creating graphics
import json
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

# @title Import MuJoCo, MJX, and Brax
from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import networks as sac_networks
from brax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
from jax import numpy as jp
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

In [ ]:
from mujoco_playground import wrapper
from mujoco_playground import registry

In [ ]:
# Fix: this playground version defaults to the Warp backend, which is not installed on Colab.
# Make every environment use the JAX (MJX) backend instead.
import importlib
importlib.reload(registry)   # start from the original functions, so this cell is safe to re-run
_orig_get_cfg = registry.get_default_config
_orig_load = registry.load

def _get_cfg_jax(env_name, _orig=_orig_get_cfg):
  cfg = _orig(env_name)
  if "impl" in cfg:
    cfg.impl = "jax"
  return cfg

def _load_jax(env_name, config=None, config_overrides=None, _orig=_orig_load):
  if config is None:
    config = _get_cfg_jax(env_name)
  return _orig(env_name, config=config, config_overrides=config_overrides)

registry.get_default_config = _get_cfg_jax
registry.load = _load_jax
print("Using JAX backend:", registry.get_default_config("Go1JoystickFlatTerrain").impl)

# Fix 2: JAX 0.11 removed jax.device_put_replicated / device_put_sharded, which brax 0.14.2 still uses.
# Drop-in replacements from the JAX docs (docs.jax.dev/en/latest/migrate_pmap.html).
import jax
import jax.numpy as jnp
import numpy as np
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

def _device_put_sharded(shards, devices):
  mesh = Mesh(np.array(devices), ("x",))
  sharding = NamedSharding(mesh, P("x"))
  return jax.tree.map(lambda *xs: jax.device_put(jnp.stack(xs), sharding), *shards)

def _device_put_replicated(x, devices):
  mesh = Mesh(np.array(devices), ("x",))
  sharding = NamedSharding(mesh, P("x"))
  return jax.tree.map(lambda y: jax.device_put(jnp.stack([y] * len(devices)), sharding), x)

jax.device_put_sharded = _device_put_sharded
jax.device_put_replicated = _device_put_replicated
print("Patched jax.device_put_replicated / device_put_sharded")


In [ ]:
env_name = 'Go1JoystickFlatTerrain'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)

from mujoco_playground.config import locomotion_params
ppo_params = locomotion_params.brax_ppo_config(env_name)
print(ppo_params)
print(env_cfg.reward_config)

In [ ]:
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

randomizer = registry.get_domain_randomizer(env_name)
ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **ppo_params.network_factory
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    randomization_fn=randomizer,
    progress_fn=progress
)

In [ ]:
import os
os.chdir('/kaggle/working')

# Go1 reward ablation — paste this as ONE new cell in the official locomotion Colab,
# after you have run the cells up to and including the "train_fn = functools.partial(...)" cell
# (the cell right after "### Train"). You can skip the original training cell.

import json, os
os.makedirs("results", exist_ok=True)
os.makedirs("videos", exist_ok=True)

NUM_TIMESTEPS = 50_000_000   # lower = faster. Try 20_000_000 first to check everything runs.

# Each experiment changes some reward weights. Default weights are printed by `env_cfg`.
EXPERIMENTS = {
    "baseline": {},
    "no_feet_rewards": {"feet_air_time": 0.0, "feet_clearance": 0.0,
                        "feet_height": 0.0, "feet_slip": 0.0},
    "energy_x10": {"energy": -0.01, "torques": -0.002},
    "no_orientation": {"orientation": 0.0},
}


def make_cfg(overrides):
  cfg = registry.get_default_config(env_name)
  for term, value in overrides.items():
    assert term in cfg.reward_config.scales, f"unknown reward term {term}"
    setattr(cfg.reward_config.scales, term, value)
  return cfg


def train_one(name, cfg):
  curve = []
  def progress(num_steps, metrics):
    curve.append((int(num_steps), float(metrics["eval/episode_reward"])))
    print(f"[{name}] {num_steps:,} steps  reward={curve[-1][1]:.2f}")
  params_dict = dict(ppo_training_params)
  params_dict["num_timesteps"] = NUM_TIMESTEPS
  fn = functools.partial(ppo.train, **params_dict, network_factory=network_factory,
                         randomization_fn=randomizer, progress_fn=progress)
  make_inf, params, _ = fn(environment=registry.load(env_name, config=cfg),
                           eval_env=registry.load(env_name, config=cfg),
                           wrap_env_fn=wrapper.wrap_for_brax_training)
  return make_inf, params, curve


def _rollout(ev, policy, cmd, steps, seed=0):
  reset, step = jax.jit(ev.reset), jax.jit(ev.step)
  rng = jax.random.PRNGKey(seed)
  command = jp.array(cmd)
  state = reset(rng)
  state.info["command"] = command
  traj, vel_err, power, fell, obs_cmd = [], [], [], False, []
  start_xy = np.array(state.data.qpos[:2])
  for t in range(steps):
    rng, key = jax.random.split(rng)
    action, _ = policy(state.obs, key)
    state = step(state, action)
    state.info["command"] = command
    traj.append(state)
    v = ev.get_local_linvel(state.data)
    vel_err.append(float(jp.linalg.norm(v[:2] - command[:2])))
    power.append(float(jp.sum(jp.abs(state.data.actuator_force * state.data.qvel[6:]))))
    fell = fell or bool(state.done)
    if t in (0, steps // 2, steps - 1):
      o = state.obs["state"] if isinstance(state.obs, dict) else state.obs
      obs_cmd.append([round(float(x), 3) for x in o[-3:]])
  dist = float(np.linalg.norm(np.array(state.data.qpos[:2]) - start_xy))
  return traj, {"mean_velocity_error": float(np.mean(vel_err)),
                "mean_power_W": float(np.mean(power)),
                "distance_m": dist,
                "avg_speed_from_position_mps": dist / (steps * ev.dt),
                "fell": fell,
                "obs_command_seen_by_policy": obs_cmd}


def evaluate(name, cfg, make_inf, params, cmd=(1.0, 0.0, 0.0), steps=500):
  # Same command and same metrics for every run, so results are comparable.
  # (The training reward itself is NOT comparable, because each run uses different weights.)
  policy = jax.jit(make_inf(params, deterministic=True))
  # (a) plain env with this run's config
  traj, m_plain = _rollout(registry.load(env_name, config=cfg), policy, cmd, steps)
  # (b) same settings as the official notebook's rollout (perturbations on but kicks off, wider command range)
  cfg2 = make_cfg(EXPERIMENTS.get(name, {}))
  cfg2.pert_config.enable = True
  cfg2.pert_config.velocity_kick = [0.0, 0.0]
  cfg2.pert_config.kick_wait_times = [5.0, 15.0]
  cfg2.command_config.a = [1.5, 0.8, 2 * jp.pi]
  ev2 = registry.load(env_name, config=cfg2)
  traj2, m_nb = _rollout(ev2, policy, cmd, steps)
  print(f"[{name}] plain eval: {m_plain}")
  print(f"[{name}] notebook-style eval: {m_nb}")
  try:   # rendering needs a working EGL setup; keep the numbers even if video fails
    frames = ev2.render(traj2[::2], camera="track", width=640, height=480)
    media.write_video(f"videos/{name}.mp4", frames, fps=1.0 / ev2.dt / 2)
  except Exception as e:
    print(f"[{name}] video not saved: {e}")
  # keep the trained weights so evaluation can be redone without retraining
  import pickle
  os.makedirs("params", exist_ok=True)
  with open(f"params/{name}.pkl", "wb") as f:
    pickle.dump(jax.tree.map(np.asarray, params), f)
  out = dict(m_nb)
  out["plain_eval"] = m_plain
  return out


def run_all(experiments, num_timesteps):
  global NUM_TIMESTEPS
  NUM_TIMESTEPS = num_timesteps
  all_results = {}
  for name, overrides in experiments.items():
    cfg = make_cfg(overrides)
    make_inf, params, curve = train_one(name, cfg)
    all_results[name] = {"overrides": overrides, "num_timesteps": num_timesteps,
                         "curve": curve, **evaluate(name, cfg, make_inf, params)}
    print(name, {k: v for k, v in all_results[name].items() if k != "curve"})
    with open("results/results.json", "w") as f:   # saved after every run
      json.dump(all_results, f, indent=2)
  print(f"\n{'experiment':18s} {'vel_error':>10s} {'power_W':>10s} {'fell':>6s}")
  for name, r in all_results.items():
    print(f"{name:18s} {r['mean_velocity_error']:10.3f} {r['mean_power_W']:10.1f} {str(r['fell']):>6s}  speed_from_position={r['avg_speed_from_position_mps']:.2f} m/s")
  return all_results


# Quick test first (one short run) to check everything works:
#   run_all({"baseline": {}}, 5_000_000)
# Then the real study:
#   results = run_all(EXPERIMENTS, NUM_TIMESTEPS)


In [ ]:
# Two experiments at the default 200M steps (about 30 min each on T4 x2).
# At 50M steps the policy had only learned to stand still, not to walk.
results = run_all({"baseline": {}, "energy_x10": EXPERIMENTS["energy_x10"]}, 200_000_000)

In [ ]:
!cd /kaggle/working && zip -qr go1_reward_study_results.zip results videos params && ls -la results videos params